In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece
!pip -q install -U fastapi uvicorn pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 6.5 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer

MODEL = "sshleifer/distilbart-cnn-12-6"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

print("Tokenizer Loaded Successfully!")

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Tokenizer Loaded Successfully!


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer # Changed model class and removed BitsAndBytesConfig

MODEL = "sshleifer/distilbart-cnn-12-6"

# Tokenizer is already loaded in the previous cell.
# tokenizer = AutoTokenizer.from_pretrained(MODEL) # Redundant, removed.

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL) # Changed model class and removed quantization config

print("✅ Model Loaded!")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

✅ Model Loaded!


In [ ]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"Running on: {device}")

app = FastAPI()


class ChatRequest(BaseModel):
    message: str


@app.get("/")
def root():
    return {
        "status": "running",
        "model": MODEL_NAME,
        "device": device
    }


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.post("/chat")
def chat(req: ChatRequest):

    try:

        text = req.message.strip()

        if len(text) == 0:
            return {"response": ""}

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.inference_mode():

            summary_ids = model.generate(

                **inputs,

                max_length=150,
                min_length=40,

                num_beams=4,

                early_stopping=True,

                no_repeat_ngram_size=3,

                length_penalty=2.0

            )

        summary = tokenizer.decode(

            summary_ids[0],

            skip_special_tokens=True

        )

        torch.cuda.empty_cache()

        return {"response": summary}

    except Exception as e:

        return {"response": f"ERROR: {str(e)}"}

Writing app.py


In [ ]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

In [ ]:
!ps -ef | grep uvicorn

root         899       1  0 14:14 ?        00:00:00 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 8000
root         900     383  0 14:14 ?        00:00:00 /bin/bash -c ps -ef | grep uvicorn
root         902     900  0 14:14 ?        00:00:00 grep uvicorn


In [ ]:
!tail -50 server.log

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading tokenizer...
Loading model...
Loading weights: 100%|██████████| 358/358 [00:00<00:00, 16397.96it/s]
INFO:     Started server process [899]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
!curl http://127.0.0.1:8000/health

{"status":"healthy"}

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000

2026-07-21T14:15:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-21T14:15:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-21T14:15:11Z INF +--------------------------------------------------------------------------------------------+
2026-07-21T14:15:11Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-21T14:15:11Z INF |  https://commissions-mining-homework-andale.trycloudfl